# M2: Depth Anything V2 + depth-ControlNet relighting

Tests whether depth-conditioned ControlNet img2img at low denoising strength can shift a Street View crop to golden hour **without changing the scene structure**.

**Input:** `data/crops/<IMAGE_ID>/view_000.png` (shared crop from M1)
**Outputs:** depth map, relit images at several strengths, comparison grid, structure-preservation numbers, rows in `docs/experiment_log.csv`.

Needs a GPU (Colab T4 is fine). Runtime > Change runtime type > GPU.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors pillow matplotlib scipy opencv-python

## 1. Settings

In [ ]:
from pathlib import Path
import json, csv, os, torch

IMAGE_ID = "shattuck_downtown_2025-03"   # CHANGE to the image ID M1 shared
VIEW = "view_000"
SEED = 42
STRENGTHS = [0.2, 0.3, 0.4]              # low denoising strengths from the plan
CONTROL_SCALES = [1.0]                   # try [0.7, 1.0] for an extra ablation
STEPS = 30
GUIDANCE = 7.5

PROMPT = ("Realistic city street at warm golden hour, natural soft sunlight, "
          "preserve original buildings, roads, vehicles and layout")
NEG_PROMPT = "extra objects, missing objects, distorted buildings, cartoon, painting, text, watermark, blurry"

DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Small-hf"
CONTROLNET_MODEL = "lllyasviel/sd-controlnet-depth"
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# Find the repo root if running inside the cloned repo, otherwise use a scratch folder (Colab)
ROOT = Path.cwd()
if (ROOT.parent / "src" / "depth.py").exists() and ROOT.name == "notebooks":
    ROOT = ROOT.parent
if (ROOT / "config" / "experiment.json").exists():
    cfg = json.load(open(ROOT / "config" / "experiment.json"))
    PROMPT, SEED = cfg.get("prompt", PROMPT), cfg.get("seed", SEED)   # keep shared settings identical
IN_PATH = ROOT / "data" / "crops" / IMAGE_ID / f"{VIEW}.png"
DEPTH_PATH = ROOT / "data" / "depth" / IMAGE_ID / f"{VIEW}.png"
OUT_DIR = ROOT / "outputs" / "controlnet" / IMAGE_ID
LOG_PATH = ROOT / "docs" / "experiment_log.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("root:", ROOT, "| device:", device, "| seed:", SEED)
print("prompt:", PROMPT)

In [ ]:
# If the crop isn't in the folder (e.g. fresh Colab), upload view_000.png from the shared Drive
if not IN_PATH.exists():
    IN_PATH.parent.mkdir(parents=True, exist_ok=True)
    from google.colab import files
    up = files.upload()
    os.rename(list(up.keys())[0], IN_PATH)
from PIL import Image
orig = Image.open(IN_PATH).convert("RGB")
print(orig.size)
orig

## 2. Depth map (Depth Anything V2 Small)

In [ ]:
from transformers import pipeline
import numpy as np
depth_pipe = pipeline("depth-estimation", model=DEPTH_MODEL, device=0 if device == "cuda" else -1)
pred = depth_pipe(orig)
depth_img = pred["depth"].convert("RGB").resize(orig.size, Image.Resampling.BILINEAR)
DEPTH_PATH.parent.mkdir(parents=True, exist_ok=True)
depth_img.save(DEPTH_PATH)
print("saved", DEPTH_PATH, depth_img.size)
depth_img

## 3. Depth-ControlNet img2img at several strengths

In [ ]:
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler
dtype = torch.float16 if device == "cuda" else torch.float32
controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL, torch_dtype=dtype)
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    BASE_MODEL, controlnet=controlnet, torch_dtype=dtype, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

OUT_DIR.mkdir(parents=True, exist_ok=True)
results = {}   # (strength, scale) -> PIL image
for s in STRENGTHS:
    for cs in CONTROL_SCALES:
        gen = torch.Generator(device=device).manual_seed(SEED)   # same seed for every run
        img = pipe(prompt=PROMPT, negative_prompt=NEG_PROMPT, image=orig, control_image=depth_img,
                   strength=s, controlnet_conditioning_scale=cs, num_inference_steps=STEPS,
                   guidance_scale=GUIDANCE, generator=gen).images[0]
        p = OUT_DIR / f"{VIEW}_s{s:.2f}_c{cs:.1f}.png"
        img.save(p); results[(s, cs)] = img
        print("saved", p)

## 4. Comparison figure

In [ ]:
import matplotlib.pyplot as plt
panels = [("Original", orig), ("Depth (Depth Anything V2)", depth_img)] + \
         [(f"ControlNet  strength={s}, scale={cs}", im) for (s, cs), im in results.items()]
fig, axs = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.6))
for ax, (t, im) in zip(axs, panels):
    ax.imshow(im); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout()
fig_path = OUT_DIR / f"{VIEW}_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
print("saved", fig_path)

## 5. Structure preservation check
Re-run depth on each output and compare with the depth of the input. Higher correlation / lower error = scene geometry better preserved. Also compares Canny edges.

In [ ]:
import cv2
def get_depth(im):
    d = np.array(depth_pipe(im)["depth"].resize(orig.size, Image.Resampling.BILINEAR)).astype(np.float32)
    return (d - d.min()) / (d.max() - d.min() + 1e-8)
def edges(im):
    return cv2.Canny(cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY), 100, 200) > 0

d0, e0 = get_depth(orig), edges(orig)
metrics = {}
for k, im in results.items():
    d1, e1 = get_depth(im), edges(im)
    e0d, e1d = cv2.dilate(e0.astype(np.uint8), np.ones((3, 3))) > 0, cv2.dilate(e1.astype(np.uint8), np.ones((3, 3))) > 0
    metrics[k] = dict(depth_corr=float(np.corrcoef(d0.ravel(), d1.ravel())[0, 1]),
                      depth_mae=float(np.abs(d0 - d1).mean()),
                      edge_iou=float((e0d & e1d).sum() / max((e0d | e1d).sum(), 1)),
                      mean_rgb_orig=np.array(orig).reshape(-1, 3).mean(0).round(1).tolist(),
                      mean_rgb_out=np.array(im).reshape(-1, 3).mean(0).round(1).tolist())
    print(k, {m: (round(v, 3) if isinstance(v, float) else v) for m, v in metrics[k].items()})

## 6. Log runs to `docs/experiment_log.csv`

In [ ]:
cols = ["image_id","view","method","model_version","prompt","seed","denoising_strength","controlnet_scale","device","result_path","notes"]
rows = []
for (s, cs), im in results.items():
    m = metrics[(s, cs)]
    rows.append(dict(image_id=IMAGE_ID, view=VIEW, method="depth-controlnet-img2img",
        model_version=f"{BASE_MODEL} + {CONTROLNET_MODEL}; depth={DEPTH_MODEL}",
        prompt=PROMPT, seed=SEED, denoising_strength=s, controlnet_scale=cs, device=device,
        result_path=str((OUT_DIR / f"{VIEW}_s{s:.2f}_c{cs:.1f}.png").relative_to(ROOT)) if (ROOT/'outputs').exists() else "",
        notes=f"depth_corr={m['depth_corr']:.3f}; depth_mae={m['depth_mae']:.3f}; edge_iou={m['edge_iou']:.3f}; TODO qualitative"))
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
new_file = not LOG_PATH.exists() or LOG_PATH.stat().st_size == 0
with open(LOG_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    if new_file: w.writeheader()
    w.writerows(rows)
print("appended", len(rows), "rows to", LOG_PATH)

## 7. Observations (fill in after looking at the images)

- **Lighting change:** (does it look golden-hour? which strength is the sweet spot?)
- **Structure preserved:** (buildings, roads, cars, signs, people unchanged? cite the depth correlation / edge IoU)
- **Failures:** (added/removed objects, warped text, colour bleeding, sky issues)
- **Chosen setting for check-in:** strength = ?, scale = ?
- **Compatibility note for IC-Light:** (does ControlNet look usable alone?)